# Notebook 01: Data Ingestion & Collection Simulation
## Smart Waste Management & Collection Optimization

This notebook demonstrates how municipal/campus bin data is gathered, formatted from Google Forms/CSVs, simulated with realistic accumulation rates, and ingested into SQLite.


In [ ]:
import sys
sys.path.append('..')
import os
import pandas as pd
import sqlite3
from src.data_loader import generate_synthetic_dataset, save_dataset_to_csv
from src.database import DatabaseManager, DEFAULT_DB_PATH


### 1. Simulate Campus / Municipal Bin Readings
Generate multi-day time-series readings with realistic refill rates.


In [ ]:
df_simulated = generate_synthetic_dataset(num_bins=25, days_of_history=4, seed=42)
print(f'Total records generated: {len(df_simulated)}')
df_simulated.head(5)


### 2. Save Raw Dataset
Export to CSV to simulate raw Google Form submissions.


In [ ]:
save_dataset_to_csv(df_simulated, '../data/sample/sample_bin_data.csv')
print('Saved to ../data/sample/sample_bin_data.csv')


### 3. Ingest into SQLite Database


In [ ]:
db = DatabaseManager('../data/database/waste_management.db')

# Prepare unique bins table records
bins_df = df_simulated.drop_duplicates(subset=['bin_id'])[['bin_id', 'location_name', 'latitude', 'longitude', 'waste_type', 'collection_frequency']]
db.bulk_insert_bins(bins_df.to_dict(orient='records'))

# Prepare readings records
readings_df = df_simulated[['bin_id', 'fill_level', 'fill_category', 'overflow_status', 'timestamp']]
db.bulk_insert_readings(readings_df.to_dict(orient='records'))
print('Database populated successfully!')
